In [1]:
# import required libraries
import pandas as pd
import numpy as np
import os
import warnings

In [2]:
# install openpyxl as pandas needs openpyxl to read .xlsx file
!pip install openpyxl

In [3]:
os.makedirs('data',   exist_ok=True)
os.makedirs('models', exist_ok=True)


## Data loading

In [4]:
# Data loading 
excel_path = r"C:\Users\HP\Downloads\edunet-IBM AIML project\online_retail_II.xlsx"
if os.path.exists(excel_path):
    df1 = pd.read_excel(excel_path, sheet_name='Year 2009-2010')
    df2 = pd.read_excel(excel_path, sheet_name='Year 2010-2011')
    df_raw =pd.concat([df1, df2], ignore_index=True)
    print("loaded local excel file")
else:
    print("No data found")

print(f'\nRaw shape: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns')
print(df_raw.head(3).to_string())

loaded local excel file

Raw shape: 1,067,371 rows and 8 columns
  Invoice StockCode                          Description  Quantity         InvoiceDate  Price  Customer ID         Country
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 2009-12-01 07:45:00   6.95      13085.0  United Kingdom
1  489434    79323P                   PINK CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom
2  489434    79323W                  WHITE CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom


In [5]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [6]:
df_raw.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [7]:
print("Cleaning data.....")
df=df_raw.copy()
print(f'Before: {len(df):,} rows')

Cleaning data.....
Before: 1,067,371 rows


## Data cleaning

In [8]:
# standardise column names
df.columns=df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer_id', 'country'],
      dtype='object')

In [9]:
df.head()

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [10]:
# Fix data types
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

In [11]:
# Remove returns, cancellations, zeroprice, missing description
df['quantity'].value_counts()

quantity
 1        294346
 2        159960
 12       121745
 6         85299
 3         72638
           ...  
 257           1
 370           1
 314           1
-1284          1
 80995         1
Name: count, Length: 1057, dtype: int64

In [12]:
# There are some quantity data shows negavite values which are invalid, remove them
before = len(df)
df = df[df['quantity'] > 0].copy()
print(f'Removed {before - len(df):,} returns (negative qty)')

Removed 22,950 returns (negative qty)


In [13]:
before = len(df)
df = df[~df['invoice'].astype(str).str.startswith('C')].copy()
print(f'Removed {before - len(df):,} cancellations')

Removed 1 cancellations


In [14]:
df['price'].value_counts()

price
1.25      102803
1.65       72507
0.85       68264
2.95       64453
0.42       44999
           ...  
370.64         1
93.27          1
268.44         1
85.10          1
133.30         1
Name: count, Length: 2280, dtype: int64

In [15]:
before = len(df)
df = df[df['price'] > 0].copy()
print(f'Removed {before - len(df):,} zero-price rows')


Removed 2,750 zero-price rows


In [16]:
before = len(df)
df = df.dropna(subset=['description']).copy()
print(f'Removed {before - len(df):,} missing descriptions')

Removed 0 missing descriptions


In [17]:
before = len(df)
df = df[df['stockcode'].astype(str).str.len() >= 5].copy()
print(f'  Removed {before - len(df):,} invalid stock codes')

  Removed 4,501 invalid stock codes


## Add useful calculated columns

In [18]:
df['revenue'] = df['quantity'] * df['price']
df['date'] = pd.to_datetime(df['invoicedate'].dt.date)
df['year'] = df['invoicedate'].dt.year
df['month'] = df['invoicedate'].dt.month
df['day_of_week'] = df['invoicedate'].dt.dayofweek  # 0=Mon, 6=Sun
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)
df['quarter'] = df['invoicedate'].dt.quarter

In [19]:
df.shape

(1037169, 15)

In [20]:
print(f'\nAfter cleaning: {len(df):,} rows ({len(df)/len(df_raw)*100:.1f}% kept)')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Products : {df["stockcode"].nunique():,}')
print(f'Countries : {df["country"].nunique()}')


After cleaning: 1,037,169 rows (97.2% kept)
Date range: 2009-12-01 → 2011-12-09
Products : 4,908
Countries : 43


## Aggregate to Daily Demand

ML models need ONE row per (product, date).
Currently we have many rows per day (one per invoice line).
GROUP BY collapses them into daily totals

In [22]:
# Use top 20 products — enough variety, faster training
top_products = df.groupby('stockcode')['quantity'].sum().nlargest(20).index.tolist()
df_top = df[df['stockcode'].isin(top_products)].copy()

# Get UK sales only (largest market, reduces noise)
df_top = df_top[df_top['country'] == 'United Kingdom'].copy()

print(f'\nUsing top 20 products, UK only: {len(df_top):,} transactions')


Using top 20 products, UK only: 37,169 transactions


In [23]:
daily = df_top.groupby(['date','stockcode','description']).agg(
    demand = ('quantity', 'sum'),
    price = ('price', 'mean'),
    revenue = ('revenue', 'sum'),
    num_invoices = ('invoice', 'nunique'),
    num_customers = ('customer_id', 'nunique'),
).reset_index()

In [26]:
# Add date columns
daily['date'] = pd.to_datetime(daily['date'])

print(f'Before filling gaps: {len(daily):,} rows')
print(f'(many dates missing because no sales on those days)')

print('\nFilling missing dates with demand=0 (KEY FIX)...')

# Full date range for all products
all_dates    = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
all_products = daily[['stockcode','description']].drop_duplicates()

# Create complete grid: every product × every date
full_index = pd.MultiIndex.from_product(
    [all_dates, all_products['stockcode'].tolist()],
    names=['date','stockcode']
)
full_df = pd.DataFrame(index=full_index).reset_index()

# Merge with actual sales
daily = full_df.merge(daily, on=['date','stockcode'], how='left')

# Fill missing sales with 0
daily['demand']        = daily['demand'].fillna(0).astype(int)
daily['revenue']       = daily['revenue'].fillna(0)
daily['num_invoices']  = daily['num_invoices'].fillna(0).astype(int)
daily['num_customers'] = daily['num_customers'].fillna(0).astype(int)


# Forward-fill price (price doesn't change on non-sale days)
daily['price'] = (daily.groupby('stockcode')['price']
                  .transform(lambda x: x.replace(0, np.nan).ffill().bfill()))

# Fill description
desc_map = all_products.set_index('stockcode')['description'].to_dict()
daily['description'] = daily['stockcode'].map(desc_map)


# Add all time features
daily['year']        = daily['date'].dt.year
daily['month']       = daily['date'].dt.month
daily['day']         = daily['date'].dt.day
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['is_weekend']  = daily['day_of_week'].isin([5,6]).astype(int)
daily['quarter']     = daily['date'].dt.quarter
daily['week']        = daily['date'].dt.isocalendar().week.astype(int)


Before filling gaps: 9,352 rows
(many dates missing because no sales on those days)

Filling missing dates with demand=0 (KEY FIX)...


In [27]:
# Placeholder columns
daily['country']      = 'United Kingdom'
daily['lost_sales']   = 0
daily['stock_level']  = 0
daily['is_promotion'] = 0
daily['discount_pct'] = 0.0

In [28]:
daily = daily.sort_values(['stockcode','date']).reset_index(drop=True)

print(f'After filling gaps : {len(daily):,} rows')
print(f'Every product now has one row per day')

After filling gaps : 22,420 rows
Every product now has one row per day


In [29]:
print('\nVerification — days per product:')
days_per_prod = daily.groupby('stockcode').size()
expected_days = len(all_dates)
print(f'Expected days per product : {expected_days}')
print(f'Min days (any product)    : {days_per_prod.min()}')
print(f'Max days (any product)    : {days_per_prod.max()}')
print(f'All equal' if days_per_prod.nunique()==1 else ' Unequal — check code')

zero_pct = (daily['demand']==0).mean()*100
print(f'\n  Zero-demand days : {zero_pct:.1f}% (normal for retail)')
print(f'  Non-zero days    : {100-zero_pct:.1f}%')


Verification — days per product:
Expected days per product : 739
Min days (any product)    : 739
Max days (any product)    : 2280
 Unequal — check code

  Zero-demand days : 33.2% (normal for retail)
  Non-zero days    : 66.8%


In [31]:
df.to_csv('C:/Users/HP/Downloads/edunet-IBM AIML project/retail_cleaned.csv', index=False)
daily.to_csv('C:/Users/HP/Downloads/edunet-IBM AIML project/retail_raw_data.csv', index=False)


In [32]:
print('\nretail_cleaned.csv   → transaction level (for EDA)')
print('data/retail_raw_data.csv  → daily aggregated (for ML)')



retail_cleaned.csv   → transaction level (for EDA)
data/retail_raw_data.csv  → daily aggregated (for ML)
